### Relative errors of sums

Givena sum of positive values:

$$
    S = \sum_{j = 1}^{N} \, x_j
$$

We're going to compute both inherited and algorithmic error. We'll see the sequential algorithm and the parallel summation.

In [1]:
# Libraries
from itertools import combinations
import sympy as smp

In [2]:
x1, x2, x3, x4, x5, x6, x7, x8 = smp.symbols("x_1:9", real = True, positive = True) # 8 total variables x_1, x_2, ..., x_8
eps1, eps2, eps3, eps4, eps5, eps6, eps7, eps8 = smp.symbols("epsilon_1:9", real = True) # RELATIVE errors of variables x_1, x_2, ..., x_8

x_list = [x1, x2, x3, x4, x5, x6, x7, x8]
eps_list = [eps1, eps2, eps3, eps4, eps5, eps6, eps7, eps8]

In [3]:
summ = sum(x_list) # Sum for N = 8
summ

x_1 + x_2 + x_3 + x_4 + x_5 + x_6 + x_7 + x_8

The relative inherited error is given by:

$$
    \epsilon_{inh} = \frac{ \left| \sum_{j = 1}^{N} \, x_j \, \frac{\partial S}{\partial x_j} \, \epsilon_j \right|}{S}
$$

In [4]:
sum([smp.Abs(x * smp.diff(summ, x, 1) * eps / summ) for x, eps in zip(x_list, eps_list)]).together() # Relative inherited error

(x_1*Abs(epsilon_1) + x_2*Abs(epsilon_2) + x_3*Abs(epsilon_3) + x_4*Abs(epsilon_4) + x_5*Abs(epsilon_5) + x_6*Abs(epsilon_6) + x_7*Abs(epsilon_7) + x_8*Abs(epsilon_8))/(x_1 + x_2 + x_3 + x_4 + x_5 + x_6 + x_7 + x_8)

This quantity is bounded. Considering that $\left| \epsilon_j \right| \leq u$:

$$
    \epsilon_{inh} \leq u
$$

Same result when summing $N$ terms

Let's compute the algorithmic error using the sequential method (the natural way to sum numbers)

In [5]:
def fl(z: smp.Symbol, err_z: smp.Symbol): # float operator
    return z * (1 + err_z)

In [6]:
eta1, eta2, eta3, eta4, eta5, eta6, eta7 = smp.symbols("eta_1:8", real = True) # Atomic relative algorithmic errors
eta_list = [eta1, eta2, eta3, eta4, eta5, eta6, eta7]

# Sequencial sum: x_1 + x_2 + ... + x_8
op1 = fl(x1 + x2, eta1)
op2 = fl(op1 + x3, eta2)
op3 = fl(op2 + x4, eta3)
op4 = fl(op3 + x5, eta4)
op5 = fl(op4 + x6, eta5)
op6 = fl(op5 + x7, eta6)
op7 = fl(op6 + x8, eta7)

op7 # Final result

(eta_7 + 1)*(x_8 + (eta_6 + 1)*(x_7 + (eta_5 + 1)*(x_6 + (eta_4 + 1)*(x_5 + (eta_3 + 1)*(x_4 + (eta_2 + 1)*(x_3 + (eta_1 + 1)*(x_1 + x_2)))))))

In [7]:
subs = {a * b: 0 for a, b in combinations(eta_list, 2)} # Linearization dictionay for subs() method
subs

{eta_1*eta_2: 0,
 eta_1*eta_3: 0,
 eta_1*eta_4: 0,
 eta_1*eta_5: 0,
 eta_1*eta_6: 0,
 eta_1*eta_7: 0,
 eta_2*eta_3: 0,
 eta_2*eta_4: 0,
 eta_2*eta_5: 0,
 eta_2*eta_6: 0,
 eta_2*eta_7: 0,
 eta_3*eta_4: 0,
 eta_3*eta_5: 0,
 eta_3*eta_6: 0,
 eta_3*eta_7: 0,
 eta_4*eta_5: 0,
 eta_4*eta_6: 0,
 eta_4*eta_7: 0,
 eta_5*eta_6: 0,
 eta_5*eta_7: 0,
 eta_6*eta_7: 0}

In [8]:
# Computing the relative algorithmic error
err_algo = (op7 - summ).expand().subs(subs) / summ
err_algo = smp.Abs(err_algo.collect(x_list))
err_algo

Abs(eta_7*x_8 + x_1*(eta_1 + eta_2 + eta_3 + eta_4 + eta_5 + eta_6 + eta_7) + x_2*(eta_1 + eta_2 + eta_3 + eta_4 + eta_5 + eta_6 + eta_7) + x_3*(eta_2 + eta_3 + eta_4 + eta_5 + eta_6 + eta_7) + x_4*(eta_3 + eta_4 + eta_5 + eta_6 + eta_7) + x_5*(eta_4 + eta_5 + eta_6 + eta_7) + x_6*(eta_5 + eta_6 + eta_7) + x_7*(eta_6 + eta_7))/(x_1 + x_2 + x_3 + x_4 + x_5 + x_6 + x_7 + x_8)

This quantity has upper bound:

$$
    \epsilon_{algo} \leq 7 \, u
$$

For a sum of $N$ terms:

$$
    \epsilon_{algo} \leq (N - 1) \, u
$$

We can do better using the parallel addition method

In [9]:
# Parallel summation:
op1 = fl(x1 + x2, eta1) # x_1 + x_2
op2 = fl(x3 + x4, eta2) # x_3 + x_4
op3 = fl(x5 + x6, eta3) # x_5 + x_6
op4 = fl(x7 + x8, eta4) # x_7 + x_8
op5 = fl(op1 + op2, eta5) # (x_1 + x_2) + (x_3 + x_4)
op6 = fl(op3 + op4, eta6) # (x_5 + x_6) + (x_7 + x_8)
op7 = fl(op5 + op6, eta7) # Total

op7 # Final result

(eta_7 + 1)*((eta_5 + 1)*((eta_1 + 1)*(x_1 + x_2) + (eta_2 + 1)*(x_3 + x_4)) + (eta_6 + 1)*((eta_3 + 1)*(x_5 + x_6) + (eta_4 + 1)*(x_7 + x_8)))

In [10]:
# Reative algorithmic error
err_algo = (op7 - summ).expand().subs(subs) / summ
err_algo = smp.Abs(err_algo.collect(x_list))
err_algo

Abs(x_1*(eta_1 + eta_5 + eta_7) + x_2*(eta_1 + eta_5 + eta_7) + x_3*(eta_2 + eta_5 + eta_7) + x_4*(eta_2 + eta_5 + eta_7) + x_5*(eta_3 + eta_6 + eta_7) + x_6*(eta_3 + eta_6 + eta_7) + x_7*(eta_4 + eta_6 + eta_7) + x_8*(eta_4 + eta_6 + eta_7))/(x_1 + x_2 + x_3 + x_4 + x_5 + x_6 + x_7 + x_8)

The relative algorithmic error obtained is bounded by:

$$
    \epsilon_{algo} \leq 3 \, u
$$

A huge improvement. For a sum of $N$ terms (with $N = 2^{p}$ with $p$ a natural number), we have:

$$
    \epsilon_{algo} \leq u \, \log_{2}(N)
$$